# PROCESAMIENTO DE  DATOS

In [2]:
import sys
import pandas as pd  
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline, Pipeline 
from sklearn.compose import ColumnTransformer, make_column_selector  
from feature_engine.selection import DropFeatures 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
import os

In [4]:
sys.path.insert(0, '../')
from src.funciones import delete_accents, one_hot_encoding

In [5]:
from src.parametros import one_hot_encoder

In [6]:
#load train data
train= pd.read_excel('../data/raw/dataTC.xlsx')
train

,ID,Edad,Tipo_Trabajo,Estado_Civil,Educacion,mora,Vivienda,Consumo,Contacto,Mes,...,Campana,Dias_Ultima_Camp,No_Contactos,Resultado_Anterior,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,1,57,servicios,casado,bachillerato,NaN,no,no,telefono fijo,may,...,1,999,0,sin contacto,1.1,93994,-36.4,4857,5191,0
1,2,37,servicios,casado,bachillerato,no,si,no,telefono fijo,may,...,1,999,0,sin contacto,1.1,93994,-36.4,4857,5191,0
2,3,40,administrador negocio,casado,primaria,no,no,no,telefono fijo,may,...,1,999,0,sin contacto,1.1,93994,-36.4,4857,5191,0
3,4,56,servicios,casado,bachillerato,no,no,si,telefono fijo,may,...,1,999,0,sin contacto,1.1,93994,-36.4,4857,5191,0
4,7,25,servicios,single,bachillerato,no,si,no,telefono fijo,may,...,1,999,0,sin contacto,1.1,93994,-36.4,4857,5191,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23094,32963,29,desempleado,soltero,preescolar,no,si,no,celular,nov,...,1,9,1,satisfactorio,-1.1,94767,-50.8,1028,4964,0
23095,32964,46,empleado,casado,Estudiante Universitario,no,no,no,celular,nov,...,1,999,0,sin contacto,-1.1,94767,-50.8,1028,4964,0
23096,32965,56,pensionado,casado,pregrado,no,si,no,celular,nov,...,2,999,0,sin contacto,-1.1,94767,-50.8,1028,4964,0
23097,32966,44,tecnico,casado,Estudiante Universitario,no,no,no,celular,nov,...,1,999,0,sin contacto,-1.1,94767,-50.8,1028,4964,1


In [7]:
#rename columns
train.rename(columns=lambda x: x.lower().replace(" ", "_"), inplace=True)

In [20]:
ids = train['id'].copy()

In [8]:
#feactures
X= train.drop("y", axis=1)
#target
y= train["y"]

Pipeline

In [9]:
pipe = Pipeline([
    # Paso 1: Eliminar columnas
    ('drop_columns', DropFeatures(['id'])),

    # Paso 2: Transformación de columnas
    ('transformer', ColumnTransformer([
        # Paso 2.1: Características numéricas
        ('num', make_pipeline(MinMaxScaler()),
         make_column_selector(dtype_include='number')),
        # Paso 2.2: Características categóricas
        ('cat', make_pipeline(
            SimpleImputer(strategy='most_frequent')
        ),
         make_column_selector(dtype_include='object'))
    ]))
])

In [10]:
#pipe fitting
pipe.fit(X, y)

Pipeline(steps=[('drop_columns', DropFeatures(features_to_drop=['id'])),
                ('transformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('minmaxscaler',
                                                                   MinMaxScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x17d63ef60>),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x17d63dee0>)]))])

In [11]:
X_train_transformed = pipe.transform(X)

In [12]:
#pipe features
features= pipe.named_steps['transformer'].get_feature_names_out()
pd.Series(features)

0                   num__edad
1                num__campana
2       num__dias_ultima_camp
3           num__no_contactos
4           num__emp_var_rate
5         num__cons_price_idx
6          num__cons_conf_idx
7              num__euribor3m
8            num__nr_employed
9           cat__tipo_trabajo
10          cat__estado_civil
11             cat__educacion
12                  cat__mora
13              cat__vivienda
14               cat__consumo
15              cat__contacto
16                   cat__mes
17                   cat__dia
18    cat__resultado_anterior
dtype: object

In [13]:
#transformed dataframe
X_train_transformed = pd.DataFrame(X_train_transformed, columns=features)
X_train_transformed

,num__edad,num__campana,num__dias_ultima_camp,num__no_contactos,num__emp_var_rate,num__cons_price_idx,num__cons_conf_idx,num__euribor3m,num__nr_employed,cat__tipo_trabajo,cat__estado_civil,cat__educacion,cat__mora,cat__vivienda,cat__consumo,cat__contacto,cat__mes,cat__dia,cat__resultado_anterior
0,0.506494,0.0,1.0,0.0,0.9375,0.991835,0.60251,0.962728,0.859848,servicios,casado,bachillerato,no,no,no,telefono fijo,may,mon,sin contacto
1,0.246753,0.0,1.0,0.0,0.9375,0.991835,0.60251,0.962728,0.859848,servicios,casado,bachillerato,no,si,no,telefono fijo,may,mon,sin contacto
2,0.285714,0.0,1.0,0.0,0.9375,0.991835,0.60251,0.962728,0.859848,administrador negocio,casado,primaria,no,no,no,telefono fijo,may,mon,sin contacto
3,0.493506,0.0,1.0,0.0,0.9375,0.991835,0.60251,0.962728,0.859848,servicios,casado,bachillerato,no,no,si,telefono fijo,may,mon,sin contacto
4,0.090909,0.0,1.0,0.0,0.9375,0.991835,0.60251,0.962728,0.859848,servicios,single,bachillerato,no,si,no,telefono fijo,may,mon,sin contacto
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23094,0.142857,0.0,0.009009,0.166667,0.479167,1.0,0.0,0.203608,0.0,desempleado,soltero,preescolar,no,si,no,celular,nov,fri,satisfactorio
23095,0.363636,0.0,1.0,0.0,0.479167,1.0,0.0,0.203608,0.0,empleado,casado,Estudiante Universitario,no,no,no,celular,nov,fri,sin contacto
23096,0.493506,0.02439,1.0,0.0,0.479167,1.0,0.0,0.203608,0.0,pensionado,casado,pregrado,no,si,no,celular,nov,fri,sin contacto
23097,0.337662,0.0,1.0,0.0,0.479167,1.0,0.0,0.203608,0.0,tecnico,casado,Estudiante Universitario,no,no,no,celular,nov,fri,sin contacto


In [14]:
delete_accents = delete_accents(X_train_transformed)
one_hot_encoded = one_hot_encoding(delete_accents, one_hot_encoder)
one_hot_encoded

,num__edad,num__campana,num__dias_ultima_camp,num__no_contactos,num__emp_var_rate,num__cons_price_idx,num__cons_conf_idx,num__euribor3m,num__nr_employed,cat__tipo_trabajo_Vive de los arriendos,...,cat__mes_jun,cat__mes_may,cat__mes_otro,cat__dia_mon,cat__dia_otro,cat__dia_thu,cat__dia_tue,cat__dia_wed,cat__resultado_anterior_otro,cat__resultado_anterior_sin contacto
0,0.5064935064935066,0.0,1.0,0.0,0.9375,0.9918351395314448,0.6025104602510458,0.9627279936558287,0.8598484848484844,0,...,0,1,0,1,0,0,0,0,0,1
1,0.24675324675324678,0.0,1.0,0.0,0.9375,0.9918351395314448,0.6025104602510458,0.9627279936558287,0.8598484848484844,0,...,0,1,0,1,0,0,0,0,0,1
2,0.28571428571428575,0.0,1.0,0.0,0.9375,0.9918351395314448,0.6025104602510458,0.9627279936558287,0.8598484848484844,0,...,0,1,0,1,0,0,0,0,0,1
3,0.4935064935064935,0.0,1.0,0.0,0.9375,0.9918351395314448,0.6025104602510458,0.9627279936558287,0.8598484848484844,0,...,0,1,0,1,0,0,0,0,0,1
4,0.09090909090909088,0.0,1.0,0.0,0.9375,0.9918351395314448,0.6025104602510458,0.9627279936558287,0.8598484848484844,0,...,0,1,0,1,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23094,0.14285714285714285,0.0,0.009009009009009009,0.16666666666666666,0.4791666666666667,1.0,0.0,0.2036082474226804,0.0,0,...,0,0,1,0,1,0,0,0,1,0
23095,0.3636363636363637,0.0,1.0,0.0,0.4791666666666667,1.0,0.0,0.2036082474226804,0.0,0,...,0,0,1,0,1,0,0,0,0,1
23096,0.4935064935064935,0.024390243902439025,1.0,0.0,0.4791666666666667,1.0,0.0,0.2036082474226804,0.0,0,...,0,0,1,0,1,0,0,0,0,1
23097,0.3376623376623377,0.0,1.0,0.0,0.4791666666666667,1.0,0.0,0.2036082474226804,0.0,0,...,0,0,1,0,1,0,0,0,0,1


In [21]:
dataTC_processed = pd.concat([ids, one_hot_encoded, y], axis=1)

**Guardar DataFrame en formato .csv**

In [22]:
#train_transformed.to_csv('train_processed.csv', index=False)
dataTC_processed.to_csv(os.path.join('../data/processed', 'dataTC_processed.csv'), index=False)

In [23]:
dataTC_processed

,id,num__edad,num__campana,num__dias_ultima_camp,num__no_contactos,num__emp_var_rate,num__cons_price_idx,num__cons_conf_idx,num__euribor3m,num__nr_employed,...,cat__mes_may,cat__mes_otro,cat__dia_mon,cat__dia_otro,cat__dia_thu,cat__dia_tue,cat__dia_wed,cat__resultado_anterior_otro,cat__resultado_anterior_sin contacto,y
0,1,0.5064935064935066,0.0,1.0,0.0,0.9375,0.9918351395314448,0.6025104602510458,0.9627279936558287,0.8598484848484844,...,1,0,1,0,0,0,0,0,1,0
1,2,0.24675324675324678,0.0,1.0,0.0,0.9375,0.9918351395314448,0.6025104602510458,0.9627279936558287,0.8598484848484844,...,1,0,1,0,0,0,0,0,1,0
2,3,0.28571428571428575,0.0,1.0,0.0,0.9375,0.9918351395314448,0.6025104602510458,0.9627279936558287,0.8598484848484844,...,1,0,1,0,0,0,0,0,1,0
3,4,0.4935064935064935,0.0,1.0,0.0,0.9375,0.9918351395314448,0.6025104602510458,0.9627279936558287,0.8598484848484844,...,1,0,1,0,0,0,0,0,1,0
4,7,0.09090909090909088,0.0,1.0,0.0,0.9375,0.9918351395314448,0.6025104602510458,0.9627279936558287,0.8598484848484844,...,1,0,1,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23094,32963,0.14285714285714285,0.0,0.009009009009009009,0.16666666666666666,0.4791666666666667,1.0,0.0,0.2036082474226804,0.0,...,0,1,0,1,0,0,0,1,0,0
23095,32964,0.3636363636363637,0.0,1.0,0.0,0.4791666666666667,1.0,0.0,0.2036082474226804,0.0,...,0,1,0,1,0,0,0,0,1,0
23096,32965,0.4935064935064935,0.024390243902439025,1.0,0.0,0.4791666666666667,1.0,0.0,0.2036082474226804,0.0,...,0,1,0,1,0,0,0,0,1,0
23097,32966,0.3376623376623377,0.0,1.0,0.0,0.4791666666666667,1.0,0.0,0.2036082474226804,0.0,...,0,1,0,1,0,0,0,0,1,1
